# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a Croissant dataset using the `mlcroissant` library. We use the FAIR^2 dataset detailing protein abundance and modifications in human samples, accessible via a Croissant schema URL.

### Dataset Source
The dataset source is provided via the Croissant schema URL below:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant object
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata fields
metadata = dataset.metadata
print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {getattr(metadata, 'identifier', '')}\n")
print(f"Published Date: {getattr(metadata, 'datePublished', '')}")

## 2. Data Overview
Let's review available record sets and their fields. We'll extract and print each record set's `@id`, its label (if available), and list its fields (by `@id`).

All access to record sets and fields is done by their `@id`.

In [ ]:
# Find all record sets in the dataset by their @id
record_sets = list(dataset.record_sets)

for rs in record_sets:
    print(f"Record Set: @id = {rs.id}")
    print(f"    label: {getattr(rs, 'name', '')}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - @id: {field.id}, label: {getattr(field, 'name', '')}, type: {getattr(field, 'data_type', '')}")
    print()

## 3. Data Extraction
We now load data from a specific record set into a DataFrame for analysis. Choose the major data table from the overview above (usually the main table with protein quantification, etc).

*Note:* All entities are referenced by their Croissant `@id`.


In [ ]:
# Extract all record set @ids for easy reference
all_record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"All available RecordSets (@id): {all_record_set_ids}")

# For this dataset, we'll use the first record set as the main table
main_record_set_id = all_record_set_ids[0]
print(f"Selected main record set for analysis: {main_record_set_id}")

# Load records for each record set into pandas DataFrames
dataframes = {}
for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for RecordSet {rs_id} with shape {df.shape}")

# Show columns for the main record set and preview
print(f"\nColumns for record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform typical EDA operations: filtering by numeric field, normalizing, and grouping. All references are by field `@id`.


In [ ]:
# Find all numeric fields (Float/Integer/etc) from main record set
main_record_set = [rs for rs in dataset.record_sets if rs.id == main_record_set_id][0]
numeric_fields = [f.id for f in main_record_set.fields if getattr(f, "data_type", "").lower() in ["float", "integer", "number"]]
print(f"Numeric fields (by @id): {numeric_fields}")

# Pick the first numeric field as an example
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise Exception('No numeric fields found in main record set.')

df = dataframes[main_record_set_id]
# Make sure numeric_field exists and can be converted to a number
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter records where numeric field > threshold
threshold = df[numeric_field_id].quantile(0.9)  # Top 10% as example threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records in {main_record_set_id} with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head(3).T)

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nFirst 3 normalized rows (field: {numeric_field_id}):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

# Find categorical/group fields (e.g. String/Text type fields)
group_candidates = [f.id for f in main_record_set.fields if getattr(f, "data_type", "").lower() in ["string", "text"]]
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"\nSelected group-by field (by @id): {group_field_id}")
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} by {group_field_id} (top 5):\n{grouped_df.head()}")
else:
    print('No textual group field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships using matplotlib. Here we plot the filtered, normalized numeric field and (if exists) show distribution by group.

In [ ]:
# Histogram of the numeric field (original and normalized)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
filtered_df[numeric_field_id].hist(ax=axs[0], bins=20, color='skyblue')
axs[0].set_title(f'Distribution of {numeric_field_id}')
filtered_df[f"{numeric_field_id}_normalized"].hist(ax=axs[1], bins=20, color='orange')
axs[1].set_title(f'Normalized {numeric_field_id}')
plt.tight_layout()
plt.show()

# If group field exists and is not too high-cardinality, boxplot
if 'group_field_id' in locals() and group_field_id in filtered_df.columns and filtered_df[group_field_id].nunique() < 20:
    filtered_df.boxplot(column=numeric_field_id, by=group_field_id, rot=90, figsize=(10,4))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and visualized the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, always referencing entities by their Croissant `@id`. We examined the structure, filtered records by quantitative fields, performed normalization, grouped by categorical attributes, and visualized patterns in protein data.

This workflow can be extended for machine learning or further downstream bioinformatics analysis.
